# Amazon Product Reviews — Text Cleaning (Week 1)

**Project:** Amazon Product Reviews Sentiment Dashboard  
**Objective:** Clean review text by removing HTML tags, punctuation, and stopwords.  
**Dataset:** [Amazon Fine Food Reviews (Kaggle)](https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews)  
**Author:** Shravani Kurlapkar

## 1. Import Libraries

In [1]:
import pandas as pd
import string
from bs4 import BeautifulSoup
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)

print('All libraries loaded successfully.')

All libraries loaded successfully.


## 2. Load the Dataset



In [3]:
FILE_PATH = '../data/raw/Reviews.csv'

df = pd.read_csv(FILE_PATH)

print(f'Dataset shape : {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Columns       : {df.columns.tolist()}')
df.head(3)

Dataset shape : 1597 rows, 27 columns
Columns       : ['id', 'asins', 'brand', 'categories', 'colors', 'dateAdded', 'dateUpdated', 'dimension', 'ean', 'keys', 'manufacturer', 'manufacturerNumber', 'name', 'prices', 'reviews.date', 'reviews.doRecommend', 'reviews.numHelpful', 'reviews.rating', 'reviews.sourceURLs', 'reviews.text', 'reviews.title', 'reviews.userCity', 'reviews.userProvince', 'reviews.username', 'sizes', 'upc', 'weight']


,id,asins,brand,categories,colors,dateAdded,dateUpdated,dimension,ean,keys,...,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username,sizes,upc,weight
0,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I initially had trouble deciding between the p...,"Paperwhite voyage, no regrets!",NaN,NaN,Cristina M,NaN,NaN,205 grams
1,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,Allow me to preface this with a little history...,One Simply Could Not Ask For More,NaN,NaN,Ricky,NaN,NaN,205 grams
2,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,4.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I am enjoying it so far. Great for reading. Ha...,Great for those that just want an e-reader,NaN,NaN,Tedd Gardiner,NaN,NaN,205 grams


## 3. Data Exploration & Basic Cleanup

In [5]:
# Check nulls
null_count = df['reviews.text'].isnull().sum()
print(f'Null values in reviews.text column: {null_count}')

# Check duplicates
dup_count = df.duplicated().sum()
print(f'Duplicate rows: {dup_count}')

# Drop nulls and duplicates
df = df.dropna(subset=['reviews.text']).drop_duplicates().reset_index(drop=True)
print(f'Shape after cleanup: {df.shape}')

Null values in reviews.text column: 0
Duplicate rows: 0
Shape after cleanup: (1597, 27)


In [7]:
# Sample raw review to see what needs cleaning
print('--- Sample raw review ---')
print(df['reviews.text'].iloc[0])

--- Sample raw review ---
I initially had trouble deciding between the paperwhite and the voyage because reviews more or less said the same thing: the paperwhite is great, but if you have spending money, go for the voyage.Fortunately, I had friends who owned each, so I ended up buying the paperwhite on this basis: both models now have 300 ppi, so the 80 dollar jump turns out pricey the voyage's page press isn't always sensitive, and if you are fine with a specific setting, you don't need auto light adjustment).It's been a week and I am loving my paperwhite, no regrets! The touch screen is receptive and easy to use, and I keep the light at a specific setting regardless of the time of day. (In any case, it's not hard to change the setting either, as you'll only be changing the light level at a certain time of day, not every now and then while reading).Also glad that I went for the international shipping option with Amazon. Extra expense, but delivery was on time, with tracking, and I did

## 4. Define Cleaning Functions

Three cleaning steps:
1. **Remove HTML tags** — BeautifulSoup
2. **Remove punctuation** — `string.punctuation`
3. **Remove stopwords** — NLTK English stopwords

In [8]:
def remove_html(text):
    """Strip HTML tags using BeautifulSoup."""
    return BeautifulSoup(text, 'html.parser').get_text()

def remove_punctuation(text):
    """Remove all punctuation characters."""
    return text.translate(str.maketrans('', '', string.punctuation))

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    """Lowercase and remove English stopwords."""
    words = text.lower().split()
    return ' '.join(w for w in words if w not in stop_words)

print(f'Loaded {len(stop_words)} English stopwords.')
print('Cleaning functions defined.')

Loaded 198 English stopwords.
Cleaning functions defined.


## 5. Apply the Cleaning Pipeline

In [9]:
df['reviews.text'] = df['reviews.text'].astype(str)

print('Step 1/3 — Removing HTML tags...')
df['cleaned_text'] = df['reviews.text'].apply(remove_html)

print('Step 2/3 — Removing punctuation...')
df['cleaned_text'] = df['cleaned_text'].apply(remove_punctuation)

print('Step 3/3 — Removing stopwords...')
df['cleaned_text'] = df['cleaned_text'].apply(remove_stopwords)

print('Cleaning complete!')

Step 1/3 — Removing HTML tags...


C:\Users\Shravani\AppData\Local\Temp\ipykernel_118772\3106271698.py:3: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  return BeautifulSoup(text, 'html.parser').get_text()


Step 2/3 — Removing punctuation...
Step 3/3 — Removing stopwords...
Cleaning complete!


## 6. Verify — Before vs After

In [10]:
for i in range(5):
    print(f'=== Review {i+1} ===')
    print(f'BEFORE: {df["reviews.text"].iloc[i][:300]}')
    print(f'AFTER : {df["cleaned_text"].iloc[i][:300]}')
    print()

=== Review 1 ===
BEFORE: I initially had trouble deciding between the paperwhite and the voyage because reviews more or less said the same thing: the paperwhite is great, but if you have spending money, go for the voyage.Fortunately, I had friends who owned each, so I ended up buying the paperwhite on this basis: both model
AFTER : initially trouble deciding paperwhite voyage reviews less said thing paperwhite great spending money go voyagefortunately friends owned ended buying paperwhite basis models 300 ppi 80 dollar jump turns pricey voyages page press isnt always sensitive fine specific setting dont need auto light adjustm

=== Review 2 ===
BEFORE: Allow me to preface this with a little history. I am (was) a casual reader who owned a Nook Simple Touch from 2011. I've read the Harry Potter series, Girl with the Dragon Tattoo series, 1984, Brave New World, and a few other key titles. Fair to say my Nook did not get as much use as many others may
AFTER : allow preface little history c

In [11]:
df['original_len'] = df['reviews.text'].str.len()
df['cleaned_len'] = df['cleaned_text'].str.len()

avg_orig = df['original_len'].mean()
avg_clean = df['cleaned_len'].mean()
reduction = (1 - avg_clean / avg_orig) * 100

print('Average character length:')
print(f'  Original : {avg_orig:.0f}')
print(f'  Cleaned  : {avg_clean:.0f}')
print(f'  Reduction: {reduction:.1f}%')

Average character length:
  Original : 909
  Cleaned  : 580
  Reduction: 36.2%


## 7. Save the Cleaned Dataset

In [12]:
# Drop helper length columns before saving
df_out = df.drop(columns=['original_len', 'cleaned_len'])

OUTPUT_PATH = '../data/processed/Reviews_cleaned.csv'
df_out.to_csv(OUTPUT_PATH, index=False)

print(f'Saved cleaned data to {OUTPUT_PATH}')
print(f'Final shape: {df_out.shape}')

Saved cleaned data to ../data/processed/Reviews_cleaned.csv
Final shape: (1597, 28)


## 8. Next Steps

- Apply sentiment scoring with **TextBlob** and **VADER**
- Explore correlation between sentiment and star ratings
- Analyze the `HelpfulnessNumerator` / `HelpfulnessDenominator` columns
- Import processed data into Power BI for dashboard building